# In-Depth Analysis of Homomorphic Encryption Libraries
**Ben-Gurion University — Confidential Computing Course**

This notebook compares **TenSEAL** and **OpenFHE** (both using the CKKS scheme) against a plaintext baseline.

Operations benchmarked: Add, Multiply, Sum, Average, Dot Product  
Metrics: Runtime (ms), Peak Memory (KB), Approximation Error, Ciphertext Size

In [ ]:
# Install libraries (Colab is Linux — both work here)
!pip install tenseal openfhe matplotlib numpy -q

In [ ]:
import time
import tracemalloc
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import tenseal as ts
from openfhe import (
    CCParamsCKKSRNS, GenCryptoContext, PKESchemeFeature
)

print('tenseal  version:', ts.__version__)
print('Libraries loaded successfully.')

## 1. Plaintext Baseline

In [ ]:
def plaintext_benchmark(vector):
    n = len(vector)
    tracemalloc.start()
    t0 = time.perf_counter()
    plain_add = [v + v for v in vector]
    plain_mul = [v * v for v in vector]
    plain_sum = sum(vector)
    plain_avg = plain_sum / n
    plain_dot = sum(a * b for a, b in zip(vector, [v*0.5+1.0 for v in vector]))
    total_s = time.perf_counter() - t0
    mem_kb = tracemalloc.get_traced_memory()[1] / 1024
    tracemalloc.stop()
    return {'total_time_s': total_s, 'peak_mem_kb': mem_kb}

## 2. TenSEAL Benchmark (CKKS)

In [ ]:
def make_tenseal_context():
    ctx = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60],
    )
    ctx.generate_galois_keys()
    ctx.global_scale = 2 ** 40
    return ctx

def tenseal_benchmark(vector):
    ctx = make_tenseal_context()
    n = len(vector)
    vector2 = [v * 0.5 + 1.0 for v in vector]
    results = {}

    def measure(fn):
        tracemalloc.start()
        t0 = time.perf_counter()
        out = fn()
        elapsed = time.perf_counter() - t0
        mem = tracemalloc.get_traced_memory()[1] / 1024
        tracemalloc.stop()
        return out, elapsed, mem

    enc,  results['encrypt_time_s'], results['encrypt_mem_kb'] = measure(lambda: ts.ckks_vector(ctx, vector))
    enc2 = ts.ckks_vector(ctx, vector2)

    enc_add, results['add_time_s'], results['add_mem_kb']   = measure(lambda: enc + enc)
    enc_mul, results['mul_time_s'], results['mul_mem_kb']   = measure(lambda: enc * enc)
    enc_sum, results['sum_time_s'], results['sum_mem_kb']   = measure(lambda: enc.sum())
    enc_avg, results['avg_time_s'], results['avg_mem_kb']   = measure(lambda: enc.sum() * (1.0/n))
    enc_dot, results['dot_time_s'], results['dot_mem_kb']   = measure(lambda: (enc * enc2).sum())

    t0 = time.perf_counter()
    dec_add = enc_add.decrypt()
    dec_mul = enc_mul.decrypt()
    dec_sum = enc_sum.decrypt()[0]
    dec_avg = enc_avg.decrypt()[0]
    dec_dot = enc_dot.decrypt()[0]
    results['decrypt_time_s'] = time.perf_counter() - t0

    exp_add = [v+v for v in vector]
    exp_mul = [v*v for v in vector]
    exp_sum = sum(vector)
    exp_avg = exp_sum / n
    exp_dot = sum(a*b for a,b in zip(vector, vector2))

    results['add_max_error'] = max(abs(a-b) for a,b in zip(dec_add, exp_add))
    results['mul_max_error'] = max(abs(a-b) for a,b in zip(dec_mul, exp_mul))
    results['sum_error']     = abs(dec_sum - exp_sum)
    results['avg_error']     = abs(dec_avg - exp_avg)
    results['dot_error']     = abs(dec_dot - exp_dot)
    results['ciphertext_bytes'] = len(enc.serialize())
    return results

## 3. OpenFHE Benchmark (CKKS)

In [ ]:
def _next_power_of_two(n):
    p = 1
    while p < n:
        p <<= 1
    return p

def make_openfhe_context(batch_size):
    params = CCParamsCKKSRNS()
    params.SetMultiplicativeDepth(3)
    params.SetScalingModSize(50)
    params.SetBatchSize(_next_power_of_two(batch_size))
    cc = GenCryptoContext(params)
    cc.Enable(PKESchemeFeature.PKE)
    cc.Enable(PKESchemeFeature.KEYSWITCH)
    cc.Enable(PKESchemeFeature.LEVELEDSHE)
    cc.Enable(PKESchemeFeature.ADVANCEDSHE)
    keys = cc.KeyGen()
    cc.EvalMultKeyGen(keys.secretKey)
    cc.EvalSumKeyGen(keys.secretKey)
    return cc, keys

def openfhe_benchmark(vector):
    n = len(vector)
    vector2 = [v * 0.5 + 1.0 for v in vector]
    cc, keys = make_openfhe_context(n)
    results = {}

    def measure(fn):
        tracemalloc.start()
        t0 = time.perf_counter()
        out = fn()
        elapsed = time.perf_counter() - t0
        mem = tracemalloc.get_traced_memory()[1] / 1024
        tracemalloc.stop()
        return out, elapsed, mem

    def decrypt_vec(ct, length):
        p = cc.Decrypt(keys.secretKey, ct)
        p.SetLength(length)
        return p.GetRealPackedValue()

    pt  = cc.MakeCKKSPackedPlaintext(vector)
    pt2 = cc.MakeCKKSPackedPlaintext(vector2)

    ct,  results['encrypt_time_s'], results['encrypt_mem_kb'] = measure(lambda: cc.Encrypt(keys.publicKey, pt))
    ct2 = cc.Encrypt(keys.publicKey, pt2)

    ct_add, results['add_time_s'], results['add_mem_kb'] = measure(lambda: cc.EvalAdd(ct, ct))
    ct_mul, results['mul_time_s'], results['mul_mem_kb'] = measure(lambda: cc.EvalMult(ct, ct))
    ct_sum, results['sum_time_s'], results['sum_mem_kb'] = measure(lambda: cc.EvalSum(ct, n))
    ct_avg, results['avg_time_s'], results['avg_mem_kb'] = measure(lambda: cc.EvalMult(cc.EvalSum(ct, n), 1.0/n))
    ct_dot, results['dot_time_s'], results['dot_mem_kb'] = measure(lambda: cc.EvalInnerProduct(ct, ct2, n))

    t0 = time.perf_counter()
    dec_add = decrypt_vec(ct_add, n)
    dec_mul = decrypt_vec(ct_mul, n)
    dec_sum = decrypt_vec(ct_sum, 1)[0]
    dec_avg = decrypt_vec(ct_avg, 1)[0]
    dec_dot = decrypt_vec(ct_dot, 1)[0]
    results['decrypt_time_s'] = time.perf_counter() - t0

    exp_add = [v+v for v in vector]
    exp_mul = [v*v for v in vector]
    exp_sum = sum(vector)
    exp_avg = exp_sum / n
    exp_dot = sum(a*b for a,b in zip(vector, vector2))

    results['add_max_error'] = max(abs(a-b) for a,b in zip(dec_add, exp_add))
    results['mul_max_error'] = max(abs(a-b) for a,b in zip(dec_mul, exp_mul))
    results['sum_error']     = abs(dec_sum - exp_sum)
    results['avg_error']     = abs(dec_avg - exp_avg)
    results['dot_error']     = abs(dec_dot - exp_dot)
    return results

## 4. Run All Benchmarks

In [ ]:
DATA = [1.0, 2.0, 3.0, 4.0, 5.0]
print(f'Input vector: {DATA}\n')

print('Running plaintext baseline...')
pt = plaintext_benchmark(DATA)

print('Running TenSEAL benchmark...')
tenseal_res = tenseal_benchmark(DATA)

print('Running OpenFHE benchmark...')
openfhe_res = openfhe_benchmark(DATA)

print('Done.')

## 5. Comparison Tables

In [ ]:
OPS      = ['Add', 'Multiply', 'Sum', 'Average', 'Dot Product']
OP_KEYS  = ['add', 'mul', 'sum', 'avg', 'dot']

print('=' * 78)
print(f'{"Timing Comparison (ms)":^78}')
print('=' * 78)
print(f'{"Operation":<16} {"Plaintext":>12} {"TenSEAL":>12} {"OpenFHE":>12}')
print('-' * 55)

ts  = tenseal_res
fhe = openfhe_res
pt_ms = pt['total_time_s'] * 1000

for name, key in zip(['Encrypt','Add','Multiply','Sum','Average','Dot Product','Decrypt'],
                     ['encrypt','add','mul','sum','avg','dot','decrypt']):
    t_ms = ts[f'{key}_time_s'] * 1000
    f_ms = fhe[f'{key}_time_s'] * 1000
    print(f'{name:<16} {"—":>12} {t_ms:>12.3f} {f_ms:>12.3f}')

ts_total  = sum(ts[f'{k}_time_s'] for k in ['encrypt','add','mul','sum','avg','dot','decrypt']) * 1000
fhe_total = sum(fhe[f'{k}_time_s'] for k in ['encrypt','add','mul','sum','avg','dot','decrypt']) * 1000
print('-' * 55)
print(f'{"ALL (plain)":<16} {pt_ms:>12.4f} {"—":>12} {"—":>12}')
print(f'{"Total (HE)":<16} {"—":>12} {ts_total:>12.2f} {fhe_total:>12.2f}')
print(f'{"Overhead vs pt":<16} {"1x":>12} {ts_total/pt_ms:>11.0f}x {fhe_total/pt_ms:>11.0f}x')

print()
print('=' * 55)
print(f'{"Approximation Errors":^55}')
print('=' * 55)
print(f'{"Operation":<16} {"TenSEAL":>18} {"OpenFHE":>18}')
print('-' * 55)
for name, k in zip(OPS, ['add_max_error','mul_max_error','sum_error','avg_error','dot_error']):
    print(f'{name:<16} {ts[k]:>18.4e} {fhe[k]:>18.4e}')

print()
print(f'TenSEAL ciphertext size: {ts["ciphertext_bytes"]:,} bytes ({ts["ciphertext_bytes"]/1024:.1f} KB)')

## 6. Charts

In [ ]:
C_TS  = '#2196F3'
C_FHE = '#FF5722'
C_PT  = '#4CAF50'
x  = np.arange(len(OPS))
w  = 0.35

ts_times  = [ts[f'{k}_time_s']  * 1000 for k in OP_KEYS]
fhe_times = [fhe[f'{k}_time_s'] * 1000 for k in OP_KEYS]
ts_mem    = [ts[f'{k}_mem_kb']         for k in OP_KEYS]
fhe_mem   = [fhe[f'{k}_mem_kb']        for k in OP_KEYS]
ts_errs   = [ts[k]  for k in ['add_max_error','mul_max_error','sum_error','avg_error','dot_error']]
fhe_errs  = [fhe[k] for k in ['add_max_error','mul_max_error','sum_error','avg_error','dot_error']]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Timing
b1 = axes[0].bar(x - w/2, ts_times,  w, label='TenSEAL', color=C_TS,  alpha=0.85)
b2 = axes[0].bar(x + w/2, fhe_times, w, label='OpenFHE', color=C_FHE, alpha=0.85)
axes[0].bar_label(b1, fmt='%.2f', padding=2, fontsize=8)
axes[0].bar_label(b2, fmt='%.2f', padding=2, fontsize=8)
axes[0].set_title('Operation Time (ms)')
axes[0].set_xticks(x); axes[0].set_xticklabels(OPS, rotation=15, ha='right')
axes[0].set_ylabel('Time (ms)'); axes[0].legend()

# Memory
b3 = axes[1].bar(x - w/2, ts_mem,  w, label='TenSEAL', color=C_TS,  alpha=0.85)
b4 = axes[1].bar(x + w/2, fhe_mem, w, label='OpenFHE', color=C_FHE, alpha=0.85)
axes[1].bar_label(b3, fmt='%.1f', padding=2, fontsize=8)
axes[1].bar_label(b4, fmt='%.1f', padding=2, fontsize=8)
axes[1].set_title('Peak Memory (KB)')
axes[1].set_xticks(x); axes[1].set_xticklabels(OPS, rotation=15, ha='right')
axes[1].set_ylabel('KB'); axes[1].legend()

# Errors
b5 = axes[2].bar(x - w/2, ts_errs,  w, label='TenSEAL', color=C_TS,  alpha=0.85)
b6 = axes[2].bar(x + w/2, fhe_errs, w, label='OpenFHE', color=C_FHE, alpha=0.85)
axes[2].set_title('Approximation Error (log)')
axes[2].set_xticks(x); axes[2].set_xticklabels(OPS, rotation=15, ha='right')
axes[2].set_ylabel('Max Abs Error'); axes[2].set_yscale('log')
axes[2].legend(); axes[2].grid(True, which='both', axis='y', alpha=0.3)

plt.suptitle('TenSEAL vs OpenFHE — CKKS Benchmark', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved chart_comparison.png')

## 7. Scaling Benchmark (different vector sizes)

In [ ]:
SIZES = [5, 10, 50, 100, 500]
pt_scaling, ts_scaling, fhe_scaling = [], [], []

for size in SIZES:
    vec = [float(i+1) for i in range(size)]
    print(f'size={size}...', end=' ', flush=True)

    pt_r  = plaintext_benchmark(vec)
    ts_r  = tenseal_benchmark(vec)
    fhe_r = openfhe_benchmark(vec)

    pt_scaling.append(pt_r['total_time_s'] * 1000)
    ts_scaling.append(sum(ts_r[f'{k}_time_s'] for k in ['encrypt','add','mul','sum','avg','dot','decrypt']) * 1000)
    fhe_scaling.append(sum(fhe_r[f'{k}_time_s'] for k in ['encrypt','add','mul','sum','avg','dot','decrypt']) * 1000)
    print(f'pt={pt_scaling[-1]:.4f}ms  ts={ts_scaling[-1]:.1f}ms  fhe={fhe_scaling[-1]:.1f}ms')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(SIZES, ts_scaling,  'o-', color=C_TS,  label='TenSEAL', linewidth=2, markersize=6)
axes[0].plot(SIZES, fhe_scaling, 's-', color=C_FHE, label='OpenFHE', linewidth=2, markersize=6)
axes[0].set_xlabel('Vector Size'); axes[0].set_ylabel('Total Time (ms)')
axes[0].set_title('Scaling: TenSEAL vs OpenFHE'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(SIZES, pt_scaling,  '^-', color=C_PT,  label='Plaintext', linewidth=2, markersize=6)
axes[1].plot(SIZES, ts_scaling,  'o-', color=C_TS,  label='TenSEAL',   linewidth=2, markersize=6)
axes[1].plot(SIZES, fhe_scaling, 's-', color=C_FHE, label='OpenFHE',   linewidth=2, markersize=6)
axes[1].set_xlabel('Vector Size'); axes[1].set_ylabel('Total Time (ms) — log scale')
axes[1].set_title('Plaintext vs Encrypted (log scale)')
axes[1].set_yscale('log'); axes[1].legend(); axes[1].grid(True, which='both', alpha=0.3)

plt.suptitle('Performance Scaling with Vector Size', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_scaling_full.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved chart_scaling_full.png')

## 8. Healthcare Analytics Use Case

A motivating real-world scenario for homomorphic encryption:

**Setting:** A hospital wants to outsource statistical analysis of patient records (e.g., average blood pressure, weighted risk score) to a cloud provider — without exposing individual patient values.

**How HE helps:** Patient data is encrypted *before* leaving the hospital. The cloud computes statistics directly on ciphertexts. Only the hospital can decrypt the result.

Below we simulate computing an average and a weighted risk score over encrypted patient values.

In [ ]:
# Simulated patient data: blood pressure readings (mmHg)
# In a real scenario these would come from the hospital's database
patient_bp = [120.0, 135.0, 118.0, 142.0, 128.0]
risk_weights = [0.2, 0.3, 0.15, 0.25, 0.1]  # sum = 1.0

ctx = make_tenseal_context()
n   = len(patient_bp)

# Encrypt patient data
enc_bp      = ts.ckks_vector(ctx, patient_bp)
enc_weights = ts.ckks_vector(ctx, risk_weights)

# Cloud-side computation (no decryption until the result is back at the hospital)
t0 = time.perf_counter()
enc_avg_bp      = enc_bp.sum() * (1.0 / n)          # encrypted average
enc_weighted_bp = (enc_bp * enc_weights).sum()        # encrypted weighted score
cloud_time_ms   = (time.perf_counter() - t0) * 1000

# Hospital decrypts only the final result
avg_bp_result      = enc_avg_bp.decrypt()[0]
weighted_bp_result = enc_weighted_bp.decrypt()[0]

# Plaintext reference
expected_avg      = sum(patient_bp) / n
expected_weighted = sum(p * w for p, w in zip(patient_bp, risk_weights))

print('Healthcare Analytics Demo (TenSEAL CKKS)')
print('=' * 50)
print(f'Patient BP readings: {patient_bp}')
print(f'Risk weights:        {risk_weights}')
print()
print(f'Encrypted average BP    : {avg_bp_result:.4f}  (expected {expected_avg:.4f})')
print(f'Encrypted weighted score: {weighted_bp_result:.4f}  (expected {expected_weighted:.4f})')
print(f'Error in average        : {abs(avg_bp_result - expected_avg):.2e}')
print(f'Error in weighted score : {abs(weighted_bp_result - expected_weighted):.2e}')
print(f'Cloud computation time  : {cloud_time_ms:.3f} ms')
print()
print('The cloud never saw the raw patient values — only ciphertexts.')